# Image-only baseline (Baseline B)

**Question.** How well does a pretrained CNN that sees only the fundus photograph detect glaucoma, and does it hold up on a different camera and hospital (PAPILA)?

All logic lives in `src/glaucoma_project/image_model/`; this notebook only calls it and shows the results. The protocol, results and limitations are in `docs/image_model.md`. Settings are in `configs/image_model.yaml`.

The same steps run from the command line: `python -m glaucoma_project.image_model.develop`, then `python -m glaucoma_project.image_model.final --run-dir runs/image_model/<run_name>`.

In [ ]:
import pandas as pd
import yaml
from IPython.display import Image, display

from glaucoma_project import COMMON_SPLIT_DIR, REPO_ROOT
from glaucoma_project.data.audit import audit_export
from glaucoma_project.data.common_split import load_common_split
from glaucoma_project.data.effective_splits import split_counts
from glaucoma_project.image_model import final
from glaucoma_project.image_model.develop import RUNS_DIR, develop
from glaucoma_project.image_model.predict import score_photo
from glaucoma_project.runtime import pick_device

CONFIG = yaml.safe_load((REPO_ROOT / "configs" / "image_model.yaml").read_text())
CONFIG

## 1. Audit the data and look at the common split

The audit checks that the prepared export is complete, unaltered, correctly labelled and free of known patient or duplicate overlap between splits. The common split of record (`splits/`) adds a precaution: CHAKSU Bosch images with the same `P<number>` may be the same patient, so they are kept together or quarantined. Only split tables are read here, never test images.

In [ ]:
table, summary = audit_export()
partitions = load_common_split()
display(split_counts(partitions))
changes = pd.read_csv(COMMON_SPLIT_DIR / "changes.csv")
changes

## 2. Train

Two head-only warm-up epochs, then the whole network is fine-tuned. The best epoch is chosen by validation AUROC. This writes a new folder under `runs/image_model/` and refuses to reuse an existing one, so change `run_name` in the config for a new experiment.

In [ ]:
RUN_DIR = develop(CONFIG)
RUN_DIR

## 3. Validation results and operating threshold

The threshold is the one with the highest specificity that still reaches the target sensitivity on validation. Validation data chose the epoch and the threshold, so these numbers are not independent evidence.

In [ ]:
import pandas as pd

display(pd.read_csv(RUN_DIR / "validation_metrics.csv").T)
display(pd.read_csv(RUN_DIR / "validation_camera_metrics.csv")[["camera", "n", "positives", "auroc", "sensitivity", "specificity", "small_class_count"]])
for name in ("learning_curves", "validation_curves", "validation_errors"):
    display(Image(str(RUN_DIR / f"{name}.png")))

## 4. Freeze, then evaluate once on the test sets

Keep `RUN_FINAL_EVALUATION = False` while experimenting. Setting it to `True` hashes the checkpoint, common split and threshold, then evaluates CHAKSU test and PAPILA **once**. It cannot be run twice, and a changed checkpoint is refused.

In [ ]:
RUN_FINAL_EVALUATION = False
if RUN_FINAL_EVALUATION:
    final.check_saved_run(RUN_DIR)
    final.freeze_run(RUN_DIR)
    display(final.final_evaluation(RUN_DIR, pick_device())[["partition", "operating_point", "n", "positives", "auroc", "average_precision", "sensitivity", "specificity", "brier"]])
else:
    print("Final tests not run.")

## 5. Score one new photograph (optional)

Applies the same square padding and normalisation as the data pipeline. The result is an uncalibrated suspicion score, not a clinical probability. Do not use held-out images here during development.

In [ ]:
NEW_PHOTO = None  # e.g. Path("/path/to/fundus.jpg")
if NEW_PHOTO is not None:
    print(score_photo(RUN_DIR, NEW_PHOTO, pick_device()))